<a href="https://colab.research.google.com/github/Bright579523/hallucination-detection/blob/main/notebooks/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hallucination Detection - Colab Runner

This notebook sets up the environment and executes the pipeline on Google Colab GPU.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Clone the repository from GitHub
!git clone https://github.com/Bright579523/hallucination-detection.git
%cd hallucination-detection

# Install requirements
!pip install -r requirements.txt

Cloning into 'hallucination-detection'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 88 (delta 27), reused 80 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 194.49 KiB | 21.61 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/hallucination-detection


### (Returning Users) Restore from Google Drive
If you already ran the detector before, skip Steps 2-5 and just restore your saved CSV files.

In [3]:
# SKIP THIS if running for the first time!
# Only use when reopening Colab to restore previous results.
!mkdir -p /content/data
!cp /content/drive/MyDrive/hallucination_project/full_detection_results.csv /content/data/
!cp /content/drive/MyDrive/hallucination_project/hard_negative_dataset.csv /content/data/
print("Restored from Google Drive!")

Restored from Google Drive!


In [ ]:
# Download COCO val2017 directly on Colab (run once)
!mkdir -p /content/data
!wget http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /content/annotations.zip

# Extract
!unzip -q /content/val2017.zip -d /content/data/
!unzip -q /content/annotations.zip -d /content/data/

# Cleanup zip files
!rm /content/val2017.zip /content/annotations.zip

# (Optional) Save to Google Drive for future use
!mkdir -p /content/drive/MyDrive/hallucination_project/coco/
!cp -r /content/data/val2017 /content/drive/MyDrive/hallucination_project/coco/

In [ ]:
# Step 4: Build the hard-negative dataset from COCO annotations
!python data/build_dataset.py --mode full --data_dir /content/data

## Step 5: OWL-ViT Detection (Person 2)

Run the OWL-ViT model on every image-prompt pair in the dataset.
This step requires GPU and may take 10-20 minutes for 5,000 rows.

In [ ]:
# Run OWL-ViT detection on the full dataset
!python models/detector.py --mode full --data_dir /content/data

### Save Results to Google Drive
Save the CSV so you do not have to re-run the detector next time.

In [ ]:
# Save detection results to Google Drive
!mkdir -p /content/drive/MyDrive/hallucination_project/
!cp /content/data/full_detection_results.csv /content/drive/MyDrive/hallucination_project/
!cp /content/data/hard_negative_dataset.csv /content/drive/MyDrive/hallucination_project/
print("Saved to Google Drive!")

## Step 6: Hallucination Verification (Person 3)

Run both verifiers to classify predictions as genuine or hallucinated.

In [4]:
# V1: Confidence Threshold Verifier
!python verifier/threshold_verifier.py --data_dir /content/data

# V2: CLIP Cosine Similarity Verifier
!python verifier/clip_verifier.py --data_dir /content/data

Loading detection results from: /content/data/full_detection_results.csv
Loaded 5000 rows.

Running Grid Search for optimal threshold...
Best threshold: 0.14
Plot saved to: /content/data/v1_f1_vs_threshold.png

Results saved to: /content/data/v1_threshold_results.csv

=== V1 Threshold Verifier Results ===
  Optimal Threshold : 0.14
  Precision         : 0.8496
  Recall            : 0.9267
  F1-Score          : 0.8865

=== Breakdown by Negative Type ===
  positive  : F1=0.0000 (2000 samples)
  hard      : F1=0.9443 (2000 samples)
  easy      : F1=0.9955 (1000 samples)

Grid search results saved to: /content/data/v1_grid_search.csv
Loading detection results from: /content/data/full_detection_results.csv
Loaded 5000 rows.
Loading CLIP model: openai/clip-vit-base-patch32...
Using device: cuda
preprocessor_config.json: 100% 316/316 [00:00<00:00, 1.95MB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved wit

## Step 7: Evaluation & Results (Person 4)

Calculate metrics (Precision, Recall, F1, ROC-AUC) and generate plots.

In [5]:
# Run evaluation on both verifier results
!python evaluation/evaluate.py --data_dir /content/data

  HALLUCINATION DETECTION — EVALUATION REPORT

Loading V1 results from: /content/data/v1_threshold_results.csv

  --- V1: Confidence Threshold ---
  Accuracy  : 0.8576
  Precision : 0.8496
  Recall    : 0.9267
  F1-Score  : 0.8865
  Confusion matrix saved to: /content/data/v1_confusion_matrix.png

  --- V1: Threshold: Breakdown by Negative Type ---
  positive  : Acc=0.7540, Prec=0.0000, Rec=0.0000, F1=0.0000 (n=2000)
  hard      : Acc=0.8945, Prec=1.0000, Rec=0.8945, F1=0.9443 (n=2000)
  easy      : Acc=0.9910, Prec=1.0000, Rec=0.9910, F1=0.9955 (n=1000)

  FINAL COMPARISON
                  method  accuracy  precision   recall      f1
V1: Confidence Threshold    0.8576   0.849633 0.926667 0.88648

Summary saved to: /content/data/evaluation_summary.csv
Breakdown saved to: /content/data/evaluation_breakdown.csv

Evaluation complete!


In [ ]:
!python demo/app.py

Initializing models (this may take a minute)...
Using device: cuda
Loading OWL-ViT...
preprocessor_config.json: 100% 392/392 [00:00<00:00, 2.39MB/s]
The image processor of type `OwlViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
config.json: 4.42kB [00:00, 13.0MB/s]
tokenizer_config.json: 100% 775/775 [00:00<00:00, 5.43MB/s]
vocab.json: 1.06MB [00:00, 110MB/s]
merges.txt: 525kB [00:00, 123MB/s]
special_tokens_map.json: 100% 460/460 [00:00<00:00, 2.40MB/s]
model.safetensors: 100% 613M/613M [00:08<00:00, 74.6MB/s]
Loading weights: 100% 412/412 [00:00<00:00, 729.56it/s, Materializing param=owlvit.visual_projection.weight]
OwlViTForObjectDetection LOAD REPORT from: google/owlvit-base-patch32
Key                                         | Status     |  | 
------